# Процессы, события, действия

В предыдущем разделе мы рассмотрели простейший алгоритм дискретно-событийного моделирования.
Почему он простейший?
Как минимум, по одной, вполне очевидной, причине.
Функции процесса и действий не принимали никаких аргументов, работая с переменными из глобальной области видимости.
В этом разделе мы рассмотрим один способ, как такое положение дел можно исправить.
Это станет хорошим заделом для дальнейшего погружения в дискретно-событийный мир.

Пусть есть некий процесс, суть которого в том, что он печатает заданный текст через заданное время.
Напишем такую программу.
Назовём её "Отложенное эхо".

In [5]:
from dataclasses import dataclass, field


@dataclass(order=True, frozen=True)
class Timeout:
    when: float
    actions: list = field(compare=False)


def delayed_echo(text, delay):
    global events

    events.put(Timeout(
        when=now + delay,
        # Вот и основное нововведение:
        # в списке действий каждый элемент -
        # это кортеж пар функции и её аргументов
        actions=[
            (echo, (text,)),
            (delayed_echo, (text, delay))
        ]
    ))


def echo(text):
    print(f"{now}\t{text}")


def des_loop(processes, until):
    global now, events

    # processes теперь также стоит записывать
    # как список кортежей (имя_функции, её_аргументы)
    for proc, args in processes:
        # Вызываем начальные процессы
        # с передачей им их же аргументов
        proc(*args)

    while not events.empty():
        event = events.get()
        if event.when > until:
            break

        now = event.when
        
        for action, args in event.actions:
            # Теперь при вызове функции действия,
            # ей будет переданы её аргументы
            action(*args)

## "Отложенное эхо"

В предыдущем разделе класс события мы назвали просто `Event`.
На самом деле, событие (англ. event) в контексте ДСМ описывается немного по-другому (как именно, станет понятно в разделе, посвящённом библиотеке SimPy).
То, что у нас есть, имеет смысл назвать *событием типа истечения заданного времени* (англ. timeout):

In [6]:
from dataclasses import dataclass, field

@dataclass(order=True, frozen=True)
class Timeout:
    when: float
    actions: list = field(compare=False)

Разница пока только в названии.
Однако на самом деле новшество будет состоять в том, что будет храниться в списке `actions`.
В нём мы будем сохранять элементы, каждый из которых является парой вида `(имя_функции, аргументы)`.
Только и всего!

Вот как это работает.
Процесс отложенного эха можно описать так:

In [7]:
def delayed_echo(text, delay):
    global events

    events.put(Timeout(
        when=now + delay,
        # Вот и основное нововведение:
        # в списке действий каждый элемент -
        # это кортеж пар функции и её аргументов
        actions=[
            (echo, (text,)),
            (delayed_echo, (text, delay))
        ]
    ))

А действие (напечатать заданный текст) - так:

In [8]:
def echo(text):
    print(f"{now}\t{text}")

Что касается основного цикла, то он претерпит лишь небольшие изменения, связанные с тем, что из действий теперь нужно получать не только имя функции, но и её аргументы:

In [9]:
def des_loop(processes, until):
    global now, events

    # processes теперь также стоит записывать
    # как список кортежей (имя_функции, её_аргументы)
    for proc, args in processes:
        # Вызываем начальные процессы
        # с передачей им их же аргументов
        proc(*args)

    while not events.empty():
        event = events.get()
        if event.when > until:
            break

        now = event.when
        
        for action, args in event.actions:
            # Теперь при вызове функции действия,
            # ей будет переданы её аргументы
            action(*args)

У нас всё ещё остаются глобальные переменные:

In [10]:
from queue import PriorityQueue

now = 0
events = PriorityQueue()

Но они связаны исключительно с использованием подхода ДСМ.
В дальнейшем мы сможем избавиться и от них.

Запустить моделирование можно так:

In [11]:
# С помощью одной функции можно создать
# сколько угодно различных процессов
processes = [
    (delayed_echo, ("Aaaa...", 3)),
    (delayed_echo, ("Bbbb...", 2)),
    (delayed_echo, ("Oooo...", 5))
]
# Моделирование
print("Время\Эхо")
des_loop(processes, until=10)

Время\Эхо
2	Bbbb...
3	Aaaa...
4	Bbbb...
5	Oooo...
6	Aaaa...
6	Bbbb...
8	Bbbb...
9	Aaaa...
10	Oooo...
10	Bbbb...


```{note}
Обратите внимание, с помощью всего одной функции можно создать бесчисленное множество процессов, отличающихся между собой своими параметрами (текстом и задержкой в данном случае).
```

Заметьте, что во времена 6 и 10 *как бы* одновременно произошли два события: (A, B) и (C, B) соответственно.
"Как бы" выделено неспроста.
В ДСМ события не могут происходить одновременно.
На самом деле и здесь в момент времени 6 сначала случилось событие A и выполнились соответствующие ему действия, и лишь потом наступило событие B и выполнились действия для него.
Важно это понимать, чтобы не удивляться "странному" поведению более сложной модели.

```{important}
Если в очереди событий есть несколько событий с одинаковым временем наступления, то эти события (и, следовательно, их действия) будут выполнены в порядке FIFO (англ. First In First Out), т.е. в порядке добавления их в очередь.
```

Таким образом, на данный момент мы способны:

* создавать несложные дискретно-событийные модели в Python;
* избавляться от глобальных параметров процессов и действий с помощью передачи функциям процессов и действий аргументов.

Но мы только начали разогреваться.
Идём дальше!
Посмотрим на более осмысленный пример, с помощью которого в конце концов придём к парадигме генераторов событий, являющейся основой библиотеки SimPy.

## Исходный код

In [1]:
from dataclasses import dataclass, field
from typing import Callable, List, Tuple, Union
from queue import PriorityQueue


@dataclass(order=True, frozen=True)
class Timeout:
    when: Union[int, float]
    actions: List[Tuple[Callable, Tuple]] = field(compare=False)


def delayed_echo(text, delay):
    global events

    events.put(Timeout(
        when=now + delay,
        actions=[
            (echo, (text,)),
            (delayed_echo, (text, delay))
        ]
    ))


def echo(text):
    print(f"{now}\t{text}")


def des_loop(processes, until):
    global now, events

    for proc, args in processes:
        proc(*args)

    while not events.empty():
        event = events.get()
        if event.when > until:
            break

        now = event.when
        
        for action, args in event.actions:
            action(*args)


now = 0
events = PriorityQueue()

processes = [
    (delayed_echo, ("A", 3)),
    (delayed_echo, ("B", 2)),
    (delayed_echo, ("C", 5))
]

print("Время\tЭхо")
des_loop(processes, until=10)

Время	Эхо
2	B
3	A
4	B
5	C
6	A
6	B
8	B
9	A
10	C
10	B
